In [1]:
import numpy as np
import numpy.typing as npt
import pandas as pd
from tqdm import tqdm

from backend import Device, Backend, get_backend
from dataset import build_math_dataset
from model import GPT, load_model
from cugpt import GPT as cuGPT


DEVICE = Device.GPU
backend = get_backend(DEVICE)

backend.device

<Device.GPU: 1>

In [2]:
math_data = build_math_dataset(
    max_number=999,
    include_operations=["plus", "minus", "multiply", "divide"],
    seed=42,
    val_fraction=1.0,
    examples_per_operation=20_000,
)

x_data = math_data.x_val
vocab_size = len(math_data.vocab)

In [3]:
gpt = GPT(
    vocab_size=vocab_size,
    blocks=2,
    model_dim=128,
    n_heads=4,
    backend=backend,
)

assert gpt.backend is backend
load_model(gpt, "best_model.safetensors")

cugpt = cuGPT(
    vocab_size=vocab_size,
    blocks=2,
    model_dim=128,
    n_heads=4,
)
cugpt.load("best_model.safetensors")

In [4]:
def causal_mask(seq_len: int, backend: Backend):
    xp = backend.xp
    upper = xp.triu(xp.ones((seq_len, seq_len), dtype=xp.float32), k=1)
    mask = xp.where(upper > 0, -xp.inf, 0.0)
    return mask[xp.newaxis, xp.newaxis, :, :]

In [5]:
ATOL = 1e-4
RTOL = 1e-4

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
def compare_outputs(py: npt.NDArray, cpp: npt.NDArray, backend: Backend):
    py_np = backend.as_numpy(py).astype(np.float64)
    cpp_np = backend.as_numpy(cpp).astype(np.float64)

    diff = np.abs(py_np - cpp_np)

    denom = np.maximum(np.abs(py_np), 1e-8)
    rel = diff / denom

    return {
        "mae": float(np.mean(diff)),
        "rmse": float(np.sqrt(np.mean((py_np - cpp_np) ** 2))),
        "max_abs": float(np.max(diff)),
        "p99_abs": float(np.percentile(diff, 99)),
        "mean_rel": float(np.mean(rel)),
        "p99_rel": float(np.percentile(rel, 99)),
        "max_rel": float(np.max(rel)),
        "allclose": bool(np.allclose(py_np, cpp_np, atol=ATOL, rtol=RTOL)),
    }


def run_case(xb: npt.NDArray, name):
    mask = causal_mask(xb.shape[1], backend)

    py = gpt(xb, mask=mask)
    cpp = cugpt.forward(xb)

    metrics = compare_outputs(py, cpp, backend)

    py_np = backend.as_numpy(py)
    cpp_np = backend.as_numpy(cpp)

    py_argmax = np.argmax(py_np, axis=-1)
    cpp_argmax = np.argmax(cpp_np, axis=-1)

    metrics["argmax_agreement"] = float(np.mean(py_argmax == cpp_argmax))
    metrics["name"] = name
    metrics["batch"] = xb.shape[0]
    metrics["seq_len"] = xb.shape[1]
    metrics["vocab"] = xb.shape[2] if xb.ndim == 3 else vocab_size

    return metrics


def random_batch(batch_size: int, seq_len: int, backend: Backend):
    xp = backend.xp
    return xp.asarray(
        rng.integers(
            0,
            vocab_size,
            size=(batch_size, seq_len),
            dtype=np.int32,
        )
    )

In [7]:
cases = []

for seq_len in [1, 2, 3, 5, 8, 16, 32, 64, 128]:
    for batch_size in [1, 2, 8, 32]:
        xb = random_batch(batch_size, seq_len, backend)
        cases.append((xb, f"random_b{batch_size}_t{seq_len}"))

In [8]:
for seq_len in [1, 8, 32, 64]:
    xb = backend.xp.zeros((4, seq_len), dtype=backend.xp.int32)
    cases.append((xb, f"zeros_t{seq_len}"))

for seq_len in [1, 8, 32, 64]:
    token = int(rng.integers(0, vocab_size))
    xb = backend.xp.full((4, seq_len), token, dtype=backend.xp.int32)
    cases.append((xb, f"constant_token_{token}_t{seq_len}"))

for seq_len in [1, 8, 32, 64]:
    xb = backend.xp.full(
        (4, seq_len),
        vocab_size - 1,
        dtype=backend.xp.int32,
    )
    cases.append((xb, f"max_token_t{seq_len}"))

In [9]:
for i in range(100):
    batch_size = int(rng.choice([1, 2, 4, 8, 16, 32, 64]))
    seq_len = int(
        rng.choice([1, 2, 3, 4, 5, 7, 8, 15, 16, 31, 32, 63, 64, 100, 127, 128])
    )

    xb = random_batch(batch_size, seq_len, backend)
    cases.append((xb, f"random_{i:03d}_b{batch_size}_t{seq_len}"))

In [10]:
results = []

for xb, name in tqdm(cases, desc="Equivalence tests"):
    metrics = run_case(xb, name)
    results.append(metrics)

Equivalence tests: 100%|██████████| 148/148 [00:01<00:00, 87.09it/s] 


In [11]:
df = pd.DataFrame(results)

df

,mae,rmse,max_abs,p99_abs,mean_rel,p99_rel,max_rel,allclose,argmax_agreement,name,batch,seq_len,vocab
0,4.531866e-07,5.431748e-07,0.000001,0.000001,5.793763e-07,0.000004,0.000005,True,1.0,random_b1_t1,1,1,21
1,7.824357e-07,1.165870e-06,0.000004,0.000004,5.331358e-07,0.000004,0.000004,True,1.0,random_b2_t1,2,1,21
2,9.777243e-07,1.307352e-06,0.000004,0.000004,1.276364e-06,0.000026,0.000043,True,1.0,random_b8_t1,8,1,21
3,9.969816e-07,1.327381e-06,0.000005,0.000004,1.045808e-06,0.000014,0.000064,True,1.0,random_b32_t1,32,1,21
4,1.138585e-06,1.482244e-06,0.000004,0.000004,1.257566e-06,0.000015,0.000023,True,1.0,random_b1_t2,1,2,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,3.060863e-06,4.530953e-06,0.000044,0.000016,4.749062e-06,0.000052,0.011278,True,1.0,random_095_b32_t63,32,63,21
144,1.890965e-06,2.708153e-06,0.000017,0.000009,2.904977e-06,0.000041,0.000856,True,1.0,random_096_b4_t32,4,32,21
145,3.077209e-06,4.550057e-06,0.000041,0.000016,5.905553e-06,0.000049,0.036324,True,1.0,random_097_b32_t31,32,31,21
146,1.544877e-06,2.176567e-06,0.000012,0.000007,1.359216e-06,0.000014,0.000171,True,1.0,random_098_b4_t16,4,16,21


In [12]:
print(f"Cases:              {len(df)}")
print(f"Allclose passed:    {df['allclose'].sum()} / {len(df)}")
print(f"Worst MAE:          {df['mae'].max():.3e}")
print(f"Worst RMSE:         {df['rmse'].max():.3e}")
print(f"Worst max abs:      {df['max_abs'].max():.3e}")
print(f"Worst p99 abs:      {df['p99_abs'].max():.3e}")
print(f"Min argmax agree:   {df['argmax_agreement'].min():.6f}")

assert df["allclose"].all(), (
    "Python and CUDA implementations are not numerically equivalent "
    "within the configured tolerance"
)
assert df["argmax_agreement"].min() >= 0.999, "Prediction agreement is below 99.9%"

Cases:              148
Allclose passed:    148 / 148
Worst MAE:          3.533e-06
Worst RMSE:         5.399e-06
Worst max abs:      7.420e-05
Worst p99 abs:      1.903e-05
Min argmax agree:   1.000000


In [13]:
batch_size = 64
dataset_results = []

for start in tqdm(range(0, len(x_data), batch_size), desc="Dataset equivalence"):
    batch = x_data[start : start + batch_size]
    xb = backend.xp.asarray(batch, dtype=backend.xp.int32)

    mask = causal_mask(xb.shape[1], backend)

    py = gpt(xb, mask=mask)
    cpp = cugpt.forward(xb)

    metrics = compare_outputs(py, cpp, backend)

    py_np = backend.as_numpy(py)
    cpp_np = backend.as_numpy(cpp)

    metrics["argmax_agreement"] = float(
        np.mean(np.argmax(py_np, axis=-1) == np.argmax(cpp_np, axis=-1))
    )

    dataset_results.append(metrics)

Dataset equivalence: 100%|██████████| 893/893 [00:11<00:00, 75.15it/s]


In [14]:
dataset_df = pd.DataFrame(dataset_results)

dataset_df

,mae,rmse,max_abs,p99_abs,mean_rel,p99_rel,max_rel,allclose,argmax_agreement
0,0.000002,0.000003,0.000029,0.000010,0.000011,0.000036,0.228092,True,1.0
1,0.000002,0.000003,0.000048,0.000010,0.000003,0.000031,0.015992,True,1.0
2,0.000002,0.000003,0.000043,0.000011,0.000003,0.000031,0.009143,True,1.0
3,0.000002,0.000003,0.000032,0.000010,0.000004,0.000038,0.024065,True,1.0
4,0.000002,0.000003,0.000033,0.000010,0.000004,0.000036,0.003669,True,1.0
...,...,...,...,...,...,...,...,...,...
888,0.000002,0.000003,0.000029,0.000010,0.000003,0.000031,0.004901,True,1.0
889,0.000002,0.000003,0.000030,0.000011,0.000003,0.000035,0.003669,True,1.0
890,0.000002,0.000003,0.000043,0.000010,0.000004,0.000038,0.004604,True,1.0
891,0.000002,0.000003,0.000030,0.000010,0.000009,0.000031,0.089863,True,1.0


In [15]:
print(f"Cases:              {len(dataset_df)}")
print(f"Allclose passed:    {dataset_df['allclose'].sum()} / {len(dataset_df)}")
print(f"MAE mean:           {dataset_df['mae'].mean():.3e}")
print(f"MAE std:            {dataset_df['mae'].std():.3e}")
print(f"Worst MAE:          {dataset_df['mae'].max():.3e}")
print(f"Worst max abs:      {dataset_df['max_abs'].max():.3e}")
print(f"Min argmax agree:   {dataset_df['argmax_agreement'].mean():.6f}")

assert dataset_df["allclose"].all(), (
    "Python and CUDA implementations are not numerically equivalent "
    "within the configured tolerance"
)
assert (
    dataset_df["argmax_agreement"].min() >= 0.999
), "Prediction agreement is below 99.9%"

Cases:              893
Allclose passed:    893 / 893
MAE mean:           2.057e-06
MAE std:            3.620e-08
Worst MAE:          2.187e-06
Worst max abs:      7.808e-05
Min argmax agree:   1.000000
